# 06 - Frequency Modeling

**Purpose:** Select the predictors for the claim frequency model in each of the
four coverages, and fit the final Poisson GLM. Candidate variables are added by
forward stepwise selection, scored on out-of-sample Poisson deviance against the
holdout set. The distribution family is taken from 04; this notebook decides the
mean structure only.

**Input:** `data/processed/frequency_data.csv`
**Output:** Fitted frequency GLMs and their relativities, carried into
08_RatingAlgorithm.
**Depends on:** 02_Cleaned_Data, 04_Frequency_Distribution

**Specification:** Poisson GLM, log link, exposure = 1 policy-year per
student-coverage so no offset is required.

**Candidate predictors:** class_year, study, gpa, greek, gender, off_campus,
distance_to_campus, sprinklered. Excluded: student_id and name (identifiers with
no causal content), coverage (models are fitted separately by coverage),
risk_tier.


First we need to analyze what predictor variables are significant,(not analyzing name, id, and its already grouped by coverage). also not using risk_tier becuase thats applied after underwriting. Model trained only on data morked training(not holdout)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
PROC = ROOT / "data" / "processed"

freq   = pd.read_csv(PROC / "frequency_data.csv")   # 40,000 student-coverage rows
claims = pd.read_csv(PROC / "claims.csv")           # 1,819 individual claims

pd.set_option('display.max_colwidth', None)

# 'class' is a Python keyword, so patsy formulas cannot use it
freq   = freq.rename(columns={'class': 'class_year'})
claims = claims.rename(columns={'class': 'class_year'})

COVERAGES = ['Personal Property', 'Additional Living Expense',
             'Guest Medical', 'Liability']

# training only; holdout is sealed until validation
fq_train = {c: freq[(freq.coverage == c) &
                    (freq.training_holdout == 'training')].reset_index(drop=True)
            for c in COVERAGES}

cl_train = {c: claims[(claims.coverage == c) &
                      (claims.training_holdout == 'training')].reset_index(drop=True)
            for c in COVERAGES}

RATING_VARS = ['class_year', 'study', 'gpa', 'greek', 'gender',
               'off_campus', 'distance_to_campus', 'sprinklered']

pd.DataFrame({
    'exposures': {c: len(fq_train[c]) for c in COVERAGES},
    'claims':    {c: len(cl_train[c]) for c in COVERAGES},
    'frequency': {c: fq_train[c].claim_count.mean().round(4) for c in COVERAGES},
})

seperated training and holdout. renamed class to classyear to avoid python conflicts with class

Personal property

In [ ]:
pp = fq_train['Personal Property'][RATING_VARS + ['claim_count']].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    pp[c] = pp[c].astype('category')

pp.head(10)

got relevant data and added 

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# holdout frame for Personal Property, same treatment as the training frame
pp_hold = freq[(freq.coverage == 'Personal Property') &
               (freq.training_holdout == 'holdout')].copy()
for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    pp_hold[c] = pp_hold[c].astype('category')


def poisson_deviance(y, mu):
    """2 * sum[ y*log(y/mu) - (y - mu) ], with 0*log(0) = 0."""
    y   = np.asarray(y,  dtype=float)
    mu  = np.asarray(mu, dtype=float)
    pos = y > 0
    term = np.zeros_like(y)
    term[pos] = y[pos] * np.log(y[pos] / mu[pos])
    return 2.0 * np.sum(term - (y - mu))


rows = []

# baseline: intercept only
null = smf.glm('claim_count ~ 1', data=pp, family=sm.families.Poisson()).fit()
rows.append(('(null)', null.aic,
             poisson_deviance(pp_hold.claim_count, null.predict(pp_hold))))

# one model per candidate predictor
for v in RATING_VARS:
    m = smf.glm(f'claim_count ~ {v}', data=pp, family=sm.families.Poisson()).fit()
    rows.append((v, m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step1 = (pd.DataFrame(rows, columns=['predictor', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step1.round(3)

Greek is going to be the first predictor

Now to check all 2 parameter models

In [ ]:
rows = []

base = smf.glm('claim_count ~ greek', data=pp, family=sm.families.Poisson()).fit()
rows.append(('greek', base.aic,
             poisson_deviance(pp_hold.claim_count, base.predict(pp_hold))))

for v in [x for x in RATING_VARS if x != 'greek']:
    f = f'claim_count ~ greek + {v}'
    m = smf.glm(f, data=pp, family=sm.families.Poisson()).fit()
    rows.append((f'greek + {v}', m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step2 = (pd.DataFrame(rows, columns=['model', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step2.round(3)

moving foward with greek and sprinklered

In [ ]:
rows = []

base = smf.glm('claim_count ~ greek + sprinklered', data=pp,
               family=sm.families.Poisson()).fit()
rows.append(('greek + sprinklered', base.aic,
             poisson_deviance(pp_hold.claim_count, base.predict(pp_hold))))

for v in [x for x in RATING_VARS if x not in ('greek', 'sprinklered')]:
    f = f'claim_count ~ greek + sprinklered + {v}'
    m = smf.glm(f, data=pp, family=sm.families.Poisson()).fit()
    rows.append((f'greek + sprinklered + {v}', m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step3 = (pd.DataFrame(rows, columns=['model', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step3.round(3)

Analyze all 4 predictor models

In [ ]:
rows = []

base = smf.glm('claim_count ~ greek + sprinklered + gpa', data=pp,
               family=sm.families.Poisson()).fit()
rows.append(('greek + sprinklered + gpa', base.aic,
             poisson_deviance(pp_hold.claim_count, base.predict(pp_hold))))

for v in [x for x in RATING_VARS if x not in ('greek', 'sprinklered', 'gpa')]:
    f = f'claim_count ~ greek + sprinklered + gpa + {v}'
    m = smf.glm(f, data=pp, family=sm.families.Poisson()).fit()
    rows.append((f'greek + sprinklered + gpa + {v}', m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step4 = (pd.DataFrame(rows, columns=['model', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step4.round(3)

**Since the 3 predictor model has lower test deviance, we will use these 3 predictors to analyze personal property

Now for Additional Living Expenses

In [ ]:
ale = fq_train['Additional Living Expense'][RATING_VARS + ['claim_count']].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    ale[c] = ale[c].astype('category')

ale_hold = freq[(freq.coverage == 'Additional Living Expense') &
                (freq.training_holdout == 'holdout')].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    ale_hold[c] = ale_hold[c].astype('category')

Defining function to perform foward stepwise selection

In [ ]:
def forward_stepwise(train, hold, candidates):
    """Forward selection on out-of-sample Poisson deviance.
    Continues while any addition improves holdout deviance.
    Returns (selected variables, one combined table of every step)."""
    keep     = []
    m_cur    = smf.glm('claim_count ~ 1', data=train,
                       family=sm.families.Poisson()).fit()
    d_cur    = poisson_deviance(hold.claim_count, m_cur.predict(hold))
    all_rows = []
    step     = 0

    while True:
        remaining = [v for v in candidates if v not in keep]
        if not remaining:
            break
        step += 1

        rows = [{'step': step,
                 'model': ' + '.join(keep) or '(null)',
                 'train_AIC': m_cur.aic, 'holdout_deviance': d_cur}]

        results = {}
        for v in remaining:
            f = 'claim_count ~ ' + ' + '.join(keep + [v])
            m = smf.glm(f, data=train, family=sm.families.Poisson()).fit()
            d = poisson_deviance(hold.claim_count, m.predict(hold))
            results[v] = (m, d)
            rows.append({'step': step,
                         'model': ' + '.join(keep + [v]),
                         'train_AIC': m.aic, 'holdout_deviance': d})

        all_rows.extend(rows)

        best = min(results, key=lambda v: results[v][1])

        if results[best][1] >= d_cur:
            break

        keep.append(best)
        m_cur, d_cur = results[best]

    return keep, (pd.DataFrame(all_rows)
                    .sort_values('holdout_deviance')
                    .reset_index(drop=True))

Defined a function to do the foward stepwise selection automatically

In [ ]:
ale_keep, ale_path = forward_stepwise(ale, ale_hold, RATING_VARS)
ale_path.round(3)

Based on above, we will use predictors greek and sprinklered for additional living expense

Now calling the function for guest medical coverage.

In [ ]:
gm = fq_train['Guest Medical'][RATING_VARS + ['claim_count']].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    gm[c] = gm[c].astype('category')

gm_hold = freq[(freq.coverage == 'Guest Medical') &
               (freq.training_holdout == 'holdout')].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    gm_hold[c] = gm_hold[c].astype('category')

In [ ]:
gm_keep, gm_path = forward_stepwise(gm, gm_hold, RATING_VARS)

gm_path.round(3)

Based on above, we will use predictors greek, gpa, and study for guest medical

Now to analyze Liability

In [ ]:
lia = fq_train['Liability'][RATING_VARS + ['claim_count']].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    lia[c] = lia[c].astype('category')

lia_hold = freq[(freq.coverage == 'Liability') &
                (freq.training_holdout == 'holdout')].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    lia_hold[c] = lia_hold[c].astype('category')

In [ ]:
lia_keep, lia_path = forward_stepwise(lia, lia_hold, RATING_VARS)

lia_path.round(3)

IMPORTANT NOTE: the above results suggest that we should use parameters (Greek, distance to campus, gpa, sprinklered, and off-campus). However this now made obvious something we did not initially consider which is there is likely high multicollinearity between distance to campus and off-campus. 

Analyze the VIF to check for multicollinearity (Note: consider running PCA on Liability later and compare models)

In [ ]:
off = (pp.off_campus == 'Off campus').astype(float)
r   = np.corrcoef(off, pp.distance_to_campus)[0, 1]

print(f'r = {r:.4f}')
print(f'VIF = {1 / (1 - r**2):.2f}')

This VIF does not warrant removing either predictor, and thhough there is likely multicollinearity between them, evidence suggest including them both will improve model predictions 

*So we will move foward using Greek, distance to campus, gpa, sprinklered, and off-campus as out predictors for liabilty

CONCLUSIONS: coefficent table

In [ ]:
import re

def clean_term(t):
    m = re.match(r'^(.*?)\[T\.(.*)\]$', t)
    return f"{m.group(1)} = {m.group(2)}" if m else t

FREQ_SPECS = {
    'Personal Property':         ['greek', 'sprinklered', 'gpa'],
    'Additional Living Expense': ['greek', 'sprinklered'],
    'Guest Medical':             ['greek', 'gpa', 'study'],
    'Liability':                 ['greek', 'distance_to_campus', 'gpa',
                                  'sprinklered', 'off_campus'],
}

CAT_VARS = ['class_year', 'study', 'greek', 'gender', 'off_campus']
CATS     = {c: pd.Categorical(freq[c]).categories for c in CAT_VARS}

rows, FREQ_FINAL = [], {}

for c, v in FREQ_SPECS.items():
    d = fq_train[c][RATING_VARS + ['claim_count']].copy()
    for k in CAT_VARS:
        d[k] = pd.Categorical(d[k], categories=CATS[k])

    m = smf.glm('claim_count ~ ' + (' + '.join(v) if v else '1'),
                data=d, family=sm.families.Poisson()).fit()
    FREQ_FINAL[c] = m

    for t in m.params.index:
        rows.append({'coverage': c, 'term': clean_term(t),
                     'coef': m.params[t], 'pval': m.pvalues[t], 'se': m.bse[t],
                     'component': 'frequency'})

freq_coefs = pd.DataFrame(rows)[['coverage', 'term', 'coef', 'pval', 'se', 'component']]
freq_coefs.to_csv(PROC / "freq_coefficients.csv", index=False)

freq_coefs.round(4)